<a href="https://colab.research.google.com/github/ManikaNagpal/How-to-Build-an-open-source-RAG-pipeline-from-scratch-/blob/main/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade \
langchain \
langchain-community \
fastembed \
llama-parse \
groq \
flashrank \
unstructured \
langchain-classic


!pip install \
qdrant-client==1.10.1 \
langchain-qdrant==0.1.4 \
langchain-core==0.4

In [ ]:
#!pip uninstall -y qdrant-client langchain-qdrant langchain-core

#!pip install \
#qdrant-client==1.10.1 \
#langchain-qdrant==0.1.4 \
#langchain-core==0.4 # Pin to a version compatible with langchain-groq and other components

In [17]:
import os

# Set your API keys
os.environ["GROQ_API_KEY"] = ""
os.environ["LLAMA_PARSE_API_KEY"] = ""

In [ ]:
!mkdir data
!gdown 1ee-BhQiH-S9a2IkHiFbJz9eX_SfcZ5m9 -O "data/meta-earnings.pdf"

In [ ]:
import os
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=os.getenv("LLAMA_PARSE_API_KEY"),
    result_type="markdown"  # structured output
)

documents = parser.load_data("/content/data/meta-earnings.pdf")

print("Parsed documents:", len(documents))
print(documents[0].text[:500])

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LangchainDocument # Alias to avoid name collision

# Convert LlamaParse documents to Langchain's Document format
langchain_docs = []
for doc in documents:
    langchain_docs.append(LangchainDocument(page_content=doc.text, metadata=doc.metadata))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

docs = text_splitter.split_documents(langchain_docs)

print("Total chunks:", len(docs))

Total chunks: 39


In [ ]:
from langchain_community.embeddings import FastEmbedEmbeddings

embedding_model = FastEmbedEmbeddings()

# Test embedding
embedding_model.embed_query("Revenue growth")

In [ ]:
!pip install qdrant-client==1.10.1

In [ ]:
from qdrant_client import QdrantClient, models
from langchain_community.vectorstores import Qdrant # Use Qdrant from langchain_community

# Initialize Qdrant (in-memory for Colab)
qdrant_client = QdrantClient(":memory:")

collection_name = "rag_meta_docs"

# Get embedding dimension for vector configuration (FastEmbed uses 384 dimensions)
embedding_dimension = len(embedding_model.embed_query("test"))

# Create the collection before adding documents
qdrant_client.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=embedding_dimension, distance=models.Distance.COSINE),
)

# Initialize Qdrant vector store directly with the client and embedding model
vector_store = Qdrant(
    client=qdrant_client,
    embeddings=embedding_model,
    collection_name=collection_name,
)

# Add documents to the vector store
vector_store.add_documents(docs)

print("Vector store created and documents added!")

In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

In [12]:
# ---- PATCH: Add missing ModelProfile BEFORE any LangChain imports ----
import langchain_core.language_models as lm

if not hasattr(lm, "ModelProfile"):
    class ModelProfile:
        pass
    lm.ModelProfile = ModelProfile

In [13]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def generate_answer(context, question):
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {"role": "system", "content": "Answer only from the provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [14]:

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def generate_answer(context, question):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer ONLY from the provided context."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{question}"
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

# RAG pipeline
def ask(query):
    # Step 1: Retrieve relevant docs
    retrieved_docs = retriever.invoke(query)

    # Step 2: Format context
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Step 3: Generate answer using Groq
    answer = generate_answer(context, query)

    # Step 4: Print nicely
    print("\n🧠 Question:", query)
    print("\n📌 Answer:\n", answer)

In [15]:
import langchain
from langchain_core.globals import set_debug

# Ensure compatibility
if not hasattr(langchain, "debug"):
    langchain.debug = False

set_debug(False)

In [16]:
ask("What was Meta's revenue growth?")


🧠 Question: What was Meta's revenue growth?

📌 Answer:
 Meta's revenue grew by 27% year-over-year, reaching $36.46 billion in the first quarter of 2024.
